# Predict side effects from molecular graphs using https://keras.io/examples/graph/mpnn-molecular-graphs/ as a tutorial

In [11]:
import os
from pathlib import Path
import warnings

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from rdkit import Chem
from rdkit import RDLogger

warnings.filterwarnings('ignore')
RDLogger.DisableLog('rdApp.*')
    
np.random.seed(42)
tf.random.set_seed(42)

DATA_PATH = Path('..') / 'Data' / 'prepared_targets.csv'
if not DATA_PATH.exists():
    DATA_PATH = Path('SiderPipeline') / 'Data' / 'prepared_targets.csv'
if not DATA_PATH.exists():
    raise FileNotFoundError('Could not find prepared_targets.csv. Set DATA_PATH to its location.')

df = pd.read_csv(DATA_PATH)
required_columns = {'smiles'}
missing = required_columns.difference(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')

target_names = list(df.columns[4:])
if not target_names:
    raise ValueError('Expected side-effect columns after the first four metadata columns.')
targets = df[target_names].apply(pd.to_numeric, errors='coerce').fillna(0.0)
if not targets.isin([0.0, 1.0]).all().all():
    raise ValueError('Side-effect targets must contain only 0, 1, or empty values.')

# Remove target columns with no positive examples; they cannot be learned or evaluated.
positive_counts = targets.sum(axis=0)
target_names = positive_counts[positive_counts > 0].index.tolist()
if not target_names:
    raise ValueError('No side-effect target contains a positive example.')

valid_smiles = df['smiles'].notna() & df['smiles'].astype(str).str.strip().ne('')
df = df.loc[valid_smiles].reset_index(drop=True)
targets = targets.loc[valid_smiles, target_names].reset_index(drop=True).astype('float32')
print(f'Molecules: {len(df):,}')
print(f'Learnable side effects: {len(target_names):,}')
print(f'Positive labels: {int(targets.to_numpy().sum()):,}')

Molecules: 1,266
Learnable side effects: 5,970
Positive labels: 147,157


In [12]:
class Featurizer:
    def __init__(self, allowable_sets):
        self.dim = 0
        self.features_mapping = {}
        for name, values in allowable_sets.items():
            values = sorted(values, key=str)
            self.features_mapping[name] = dict(zip(values, range(self.dim, self.dim + len(values))))
            self.dim += len(values)

    def encode(self, value):
        output = np.zeros(self.dim, dtype='float32')
        for name, mapping in self.features_mapping.items():
            feature = getattr(self, name)(value)
            if feature in mapping:
                output[mapping[feature]] = 1.0
        return output

class AtomFeaturizer(Featurizer):
    def symbol(self, atom): return atom.GetSymbol()
    def n_valence(self, atom): return atom.GetTotalValence()
    def n_hydrogens(self, atom): return atom.GetTotalNumHs()
    def hybridization(self, atom): return atom.GetHybridization().name.lower()

class BondFeaturizer(Featurizer):
    def __init__(self, allowable_sets):
        super().__init__(allowable_sets)
        self.dim += 1

    def encode(self, bond):
        if bond is None:
            output = np.zeros(self.dim, dtype='float32')
            output[-1] = 1.0
            return output
        return super().encode(bond)

    def bond_type(self, bond): return bond.GetBondType().name.lower()
    def conjugated(self, bond): return bond.GetIsConjugated()

atom_featurizer = AtomFeaturizer({
    'symbol': {'B', 'Br', 'C', 'Ca', 'Cl', 'F', 'H', 'I', 'N', 'Na', 'O', 'P', 'S'},
    'n_valence': {0, 1, 2, 3, 4, 5, 6},
    'n_hydrogens': {0, 1, 2, 3, 4},
    'hybridization': {'s', 'sp', 'sp2', 'sp3'},
})
bond_featurizer = BondFeaturizer({
    'bond_type': {'single', 'double', 'triple', 'aromatic'},
    'conjugated': {True, False},
})

def molecule_from_smiles(smiles):
    molecule = Chem.MolFromSmiles(str(smiles), sanitize=False)
    if molecule is None:
        return None
    flag = Chem.SanitizeMol(molecule, catchErrors=True)
    if flag != Chem.SanitizeFlags.SANITIZE_NONE:
        Chem.SanitizeMol(molecule, sanitizeOps=Chem.SanitizeFlags.SANITIZE_ALL ^ flag)
    Chem.AssignStereochemistry(molecule, cleanIt=True, force=True)
    return molecule

def graph_from_molecule(molecule):
    atom_features, bond_features, pair_indices = [], [], []
    for atom in molecule.GetAtoms():
        atom_features.append(atom_featurizer.encode(atom))
        pair_indices.append([atom.GetIdx(), atom.GetIdx()])
        bond_features.append(bond_featurizer.encode(None))
        for neighbor in atom.GetNeighbors():
            bond = molecule.GetBondBetweenAtoms(atom.GetIdx(), neighbor.GetIdx())
            pair_indices.append([atom.GetIdx(), neighbor.GetIdx()])
            bond_features.append(bond_featurizer.encode(bond))
    return np.asarray(atom_features), np.asarray(bond_features), np.asarray(pair_indices)

def graphs_from_smiles(smiles_list):
    atom_list, bond_list, pair_list, keep = [], [], [], []
    for index, smiles in enumerate(smiles_list):
        molecule = molecule_from_smiles(smiles)
        if molecule is None or molecule.GetNumAtoms() == 0:
            continue
        atom_features, bond_features, pair_indices = graph_from_molecule(molecule)
        atom_list.append(atom_features)
        bond_list.append(bond_features)
        pair_list.append(pair_indices)
        keep.append(index)
    return (tf.ragged.constant(atom_list, dtype=tf.float32),
            tf.ragged.constant(bond_list, dtype=tf.float32),
            tf.ragged.constant(pair_list, dtype=tf.int64), keep)

permuted = np.random.permutation(len(df))
train_end, valid_end = int(len(df) * 0.8), int(len(df) * 0.9)
splits = {'train': permuted[:train_end], 'valid': permuted[train_end:valid_end], 'test': permuted[valid_end:]}
graph_data, label_data = {}, {}
for name, indices in splits.items():
    graphs = graphs_from_smiles(df.iloc[indices]['smiles'].to_numpy())
    kept_indices = indices[graphs[3]]
    graph_data[name] = graphs[:3]
    label_data[name] = targets.iloc[kept_indices].to_numpy(dtype='float32')
    splits[name] = kept_indices
    print(f'{name}: {len(kept_indices):,} molecules')

train: 1,012 molecules
valid: 127 molecules
test: 127 molecules


In [13]:
def prepare_batch(x_batch, y_batch):
    atom_features, bond_features, pair_indices = x_batch
    num_atoms = atom_features.row_lengths()
    num_bonds = bond_features.row_lengths()
    molecule_indices = tf.range(tf.shape(num_atoms)[0])
    molecule_indicator = tf.repeat(molecule_indices, num_atoms)
    gather_indices = tf.repeat(molecule_indices[:-1], num_bonds[1:])
    increment = tf.cumsum(num_atoms[:-1])
    increment = tf.pad(tf.gather(increment, gather_indices), [(num_bonds[0], 0)])
    pair_indices = pair_indices.merge_dims(0, 1).to_tensor() + increment[:, tf.newaxis]
    atom_features = atom_features.merge_dims(0, 1).to_tensor()
    bond_features = bond_features.merge_dims(0, 1).to_tensor()
    return (atom_features, bond_features, tf.cast(pair_indices, tf.int32), molecule_indicator), y_batch

def mpnn_dataset(x, y, batch_size=32, shuffle=False):
    dataset = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        dataset = dataset.shuffle(min(len(y), 1024), seed=42)
    return dataset.batch(batch_size).map(prepare_batch, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

class EdgeNetwork(layers.Layer):
    def build(self, input_shape):
        self.atom_dim, self.bond_dim = input_shape[0][-1], input_shape[1][-1]
        self.kernel = self.add_weight(shape=(self.bond_dim, self.atom_dim * self.atom_dim), initializer='glorot_uniform')
        self.bias = self.add_weight(shape=(self.atom_dim * self.atom_dim,), initializer='zeros')
    def call(self, inputs):
        atom_features, bond_features, pair_indices = inputs
        transformed_bonds = tf.reshape(tf.matmul(bond_features, self.kernel) + self.bias, (-1, self.atom_dim, self.atom_dim))
        neighbors = tf.expand_dims(tf.gather(atom_features, pair_indices[:, 1]), -1)
        messages = tf.squeeze(tf.matmul(transformed_bonds, neighbors), -1)
        return tf.math.unsorted_segment_sum(messages, pair_indices[:, 0], tf.shape(atom_features)[0])

class MessagePassing(layers.Layer):
    def __init__(self, units=64, steps=4, **kwargs):
        super().__init__(**kwargs); self.units, self.steps = units, steps
    def build(self, input_shape):
        atom_dim = input_shape[0][-1]
        self.padding = max(0, self.units - atom_dim)
        self.edge_network = EdgeNetwork()
        self.gru = layers.GRUCell(atom_dim + self.padding)
    def call(self, inputs):
        atom_features, bond_features, pair_indices = inputs
        state = tf.pad(atom_features, [(0, 0), (0, self.padding)])
        for _ in range(self.steps):
            messages = self.edge_network([state, bond_features, pair_indices])
            state, _ = self.gru(messages, [state])
        return state

class Readout(layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs); self.projection = layers.Dense(units, activation='relu')
    def call(self, inputs):
        atom_states, molecule_indicator = inputs
        count = tf.reduce_max(molecule_indicator) + 1
        pooled = tf.math.unsorted_segment_mean(atom_states, molecule_indicator, count)
        return self.projection(pooled)

def MPNNModel(atom_dim, bond_dim, target_dim, message_units=64, message_steps=4, dense_units=256):
    atom_features = layers.Input((atom_dim,), dtype='float32', name='atom_features')
    bond_features = layers.Input((bond_dim,), dtype='float32', name='bond_features')
    pair_indices = layers.Input((2,), dtype='int32', name='pair_indices')
    molecule_indicator = layers.Input((), dtype='int32', name='molecule_indicator')
    states = MessagePassing(message_units, message_steps)([atom_features, bond_features, pair_indices])
    embedding = Readout(message_units)([states, molecule_indicator])
    output = layers.Dense(dense_units, activation='relu')(embedding)
    output = layers.Dense(target_dim, activation='sigmoid', name='side_effect_probabilities')(output)
    return keras.Model([atom_features, bond_features, pair_indices, molecule_indicator], output)

x_train, y_train = graph_data['train'], label_data['train']
x_valid, y_valid = graph_data['valid'], label_data['valid']
x_test, y_test = graph_data['test'], label_data['test']
train_dataset = mpnn_dataset(x_train, y_train, shuffle=True)
valid_dataset = mpnn_dataset(x_valid, y_valid)
test_dataset = mpnn_dataset(x_test, y_test)
mpnn = MPNNModel(x_train[0][0][0].shape[0], x_train[1][0][0].shape[0], len(target_names))
mpnn.compile(optimizer=keras.optimizers.Adam(5e-4), loss='binary_crossentropy', metrics=[keras.metrics.AUC(name='micro_auc', multi_label=False)])
mpnn.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ atom_features       │ (None, 29)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bond_features       │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pair_indices        │ (None, 2)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ message_passing_1   │ (None, 64)        │     57,728 │ atom_features[0]… │
│ (MessagePassing)    │                   │            │ bond_features[0]… │
│                     │                   │            │ pair_indices[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ molecule_indicator  │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ readout_1 (Readout) │ (None, 64)        │      4,160 │ message_passing_… │
│                     │                   │            │ molecule_indicat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 256)       │     16,640 │ readout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ side_effect_probab… │ (None, 5970)      │  1,534,290 │ dense_3[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,612,818 (6.15 MB)

 Trainable params: 1,612,818 (6.15 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)]
history = mpnn.fit(train_dataset, validation_data=valid_dataset, epochs=40, verbose=2, callbacks=callbacks)
test_metrics = mpnn.evaluate(test_dataset, return_dict=True, verbose=0)
print({name: round(float(value), 4) for name, value in test_metrics.items()})

probabilities = mpnn.predict(test_dataset, verbose=0)
top_k = min(5, len(target_names))
top_indices = np.argsort(probabilities, axis=1)[:, -top_k:][:, ::-1]
for row in range(min(5, len(probabilities))):
    predictions = [(target_names[index], round(float(probabilities[row, index]), 3)) for index in top_indices[row]]
    print(f'{df.iloc[splits["test"][row]]["drug_name"]}: {predictions}')

# Save probabilities with molecule identifiers for downstream analysis.
prediction_frame = pd.DataFrame(probabilities, columns=target_names)
prediction_frame.insert(0, 'smiles', df.iloc[splits['test']]['smiles'].to_numpy())
prediction_frame.insert(0, 'drug_name', df.iloc[splits['test']]['drug_name'].to_numpy())
prediction_frame.to_csv(Path('side_effect_predictions.csv'), index=False)
print('Saved side_effect_predictions.csv')

Epoch 1/40
32/32 - 3s - 89ms/step - loss: 0.5156 - micro_auc: 0.5608 - val_loss: 0.1242 - val_micro_auc: 0.7581
Epoch 2/40
32/32 - 1s - 47ms/step - loss: 0.0720 - micro_auc: 0.8422 - val_loss: 0.0671 - val_micro_auc: 0.8768
Epoch 3/40
32/32 - 1s - 47ms/step - loss: 0.0626 - micro_auc: 0.9005 - val_loss: 0.0650 - val_micro_auc: 0.8991
Epoch 4/40
32/32 - 1s - 45ms/step - loss: 0.0618 - micro_auc: 0.9118 - val_loss: 0.0648 - val_micro_auc: 0.9040
Epoch 5/40
32/32 - 1s - 45ms/step - loss: 0.0619 - micro_auc: 0.9141 - val_loss: 0.0648 - val_micro_auc: 0.9040
Epoch 6/40
32/32 - 1s - 44ms/step - loss: 0.0622 - micro_auc: 0.9144 - val_loss: 0.0647 - val_micro_auc: 0.9046
Epoch 7/40
32/32 - 1s - 46ms/step - loss: 0.0611 - micro_auc: 0.9137 - val_loss: 0.0650 - val_micro_auc: 0.9045
Epoch 8/40
32/32 - 2s - 49ms/step - loss: 0.0623 - micro_auc: 0.9156 - val_loss: 0.0650 - val_micro_auc: 0.9048
Epoch 9/40
32/32 - 2s - 49ms/step - loss: 0.0620 - micro_auc: 0.9142 - val_loss: 0.0649 - val_micro_auc:

In [15]:
# Compare predictions with the known side effects in the held-out test set.
y_true = np.asarray(y_test, dtype='float32')
y_score = np.asarray(probabilities, dtype='float32')
if y_true.shape != y_score.shape:
    raise ValueError(f'Test labels and predictions have different shapes: {y_true.shape} vs {y_score.shape}')

threshold = 0.5
y_pred = (y_score >= threshold).astype('float32')
true_positive = np.sum(y_true * y_pred)
false_positive = np.sum((1.0 - y_true) * y_pred)
false_negative = np.sum(y_true * (1.0 - y_pred))
true_negative = np.sum((1.0 - y_true) * (1.0 - y_pred))
precision = true_positive / max(true_positive + false_positive, 1.0)
recall = true_positive / max(true_positive + false_negative, 1.0)
f1 = 2.0 * precision * recall / max(precision + recall, 1e-12)
subset_accuracy = np.mean(np.all(y_true == y_pred, axis=1))

roc_auc = keras.metrics.AUC(name='roc_auc')
pr_auc = keras.metrics.AUC(name='pr_auc', curve='PR')
roc_auc.update_state(y_true.ravel(), y_score.ravel())
pr_auc.update_state(y_true.ravel(), y_score.ravel())

top_k = min(5, y_score.shape[1])
top_indices = np.argsort(y_score, axis=1)[:, -top_k:]
top_k_hits = np.array([
    len(set(top_indices[row]).intersection(np.flatnonzero(y_true[row])))
    for row in range(len(y_true))
])
rows_with_known_effect = np.sum(y_true, axis=1) > 0
top_k_recall = np.mean(
    top_k_hits[rows_with_known_effect] / np.maximum(np.sum(y_true[rows_with_known_effect], axis=1), 1)
) if np.any(rows_with_known_effect) else 0.0

print('Held-out test metrics')
print(f'ROC-AUC (micro): {float(roc_auc.result()):.4f}')
print(f'PR-AUC (micro):  {float(pr_auc.result()):.4f}')
print(f'Precision @ {threshold}: {precision:.4f}')
print(f'Recall @ {threshold}:    {recall:.4f}')
print(f'F1 @ {threshold}:        {f1:.4f}')
print(f'Exact label-set accuracy: {subset_accuracy:.4f}')
print(f'Top-{top_k} recall:          {top_k_recall:.4f}')
print(f'Predicted positive labels: {int(y_pred.sum()):,} / {int(y_true.sum()):,} real positives')

comparison = pd.DataFrame({
    'drug_name': df.iloc[splits['test']]['drug_name'].to_numpy(),
    'smiles': df.iloc[splits['test']]['smiles'].to_numpy(),
})
for index, name in enumerate(target_names):
    comparison[f'real_{name}'] = y_true[:, index].astype('int8')
    comparison[f'predicted_{name}'] = y_score[:, index]
comparison.to_csv(Path('side_effect_prediction_comparison.csv'), index=False)

for row in range(min(5, len(y_true))):
    real = [target_names[index] for index in np.flatnonzero(y_true[row])]
    predicted = [target_names[index] for index in top_indices[row][::-1]]
    print(f'\n{comparison.iloc[row]["drug_name"]}')
    print(f'  Real side effects ({len(real)}): {real[:15]}')
    print(f'  Top-{top_k} predicted: {predicted}')
print('Saved side_effect_prediction_comparison.csv')

Held-out test metrics
ROC-AUC (micro): 0.9081
PR-AUC (micro):  0.2894
Precision @ 0.5: 0.5436
Recall @ 0.5:    0.1192
F1 @ 0.5:        0.1955
Exact label-set accuracy: 0.0000
Top-5 recall:          0.0571
Predicted positive labels: 2,921 / 13,327 real positives

zolpidem
  Real side effects (327): ['Abdominal pain', 'Gastrointestinal pain', 'Anaemia', 'Anorexia', 'Decreased appetite', 'Anxiety', 'Arrhythmia', 'Asthenia', 'Back pain', 'Bronchitis', 'Chest pain', 'Infection', 'Constipation', 'Cough', 'Depression']
  Top-5 predicted: ['Nausea', 'Dermatitis', 'Rash', 'Vomiting', 'Headache']

iloprost
  Real side effects (68): ['Back pain', 'Chest pain', 'Cough', 'Diarrhoea', 'Dizziness', 'Dysgeusia', 'Dyspnoea', 'Rash', 'Dermatitis', 'Gastrointestinal disorder', 'Headache', 'Haemorrhage', 'Haemoglobin', 'Hypersensitivity', 'Hypotension']
  Top-5 predicted: ['Nausea', 'Dermatitis', 'Rash', 'Vomiting', 'Headache']

gadoteridol
  Real side effects (76): ['Abdominal cramps', 'Abdominal pain', 